In [1]:
# =========================================
# IMPORT LIBRARIES
# =========================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import boto3
from botocore.client import Config

In [2]:
# =========================================
# INIT SPARK SESSION
# =========================================

spark = SparkSession.builder \
    .appName("SV3_Altcoins_ETL") \
    .getOrCreate()

print("Spark Started Successfully")

# =========================================
# MINIO S3A CONFIG
# =========================================

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "admin")
hadoop_conf.set("fs.s3a.secret.key", "password123")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

hadoop_conf.set(
    "fs.s3a.aws.credentials.provider",
    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
)

print("MinIO Configuration Completed")

Spark Started Successfully
MinIO Configuration Completed


In [3]:
# =========================================
# LOAD RAW DATA FROM MINIO
# =========================================

df = spark.read.csv(
    "s3a://crypto-raw-data/altcoins_500d.csv",
    header=True,
    inferSchema=True
)

print("Raw Dataset Loaded")

print("Total Rows:", df.count())

df.show(20, False)

df.printSchema()

Raw Dataset Loaded
Total Rows: 7000
+-------+----------+------+------+------+------+--------------+
|symbol |timestamp |open  |high  |low   |close |volume        |
+-------+----------+------+------+------+------+--------------+
|ETH/USD|2025-01-27|3228.0|3251.5|3024.0|3181.9|16821.74301423|
|ETH/USD|2025-01-28|3181.7|3224.9|3039.0|3075.8|4970.19146861 |
|ETH/USD|2025-01-29|3077.6|3181.1|3055.0|3114.2|4377.18334445 |
|ETH/USD|2025-01-30|3114.3|3283.1|3092.2|3247.8|5651.86284548 |
|ETH/USD|2025-01-31|3247.2|3437.9|3214.0|3300.1|8634.67643534 |
|ETH/USD|2025-02-01|3299.5|3331.5|3101.6|3116.8|3380.51704713 |
|ETH/USD|2025-02-02|3115.0|3162.5|2751.0|2869.1|14960.17996076|
|ETH/USD|2025-02-03|2869.6|2923.0|2118.0|2883.2|46609.82125989|
|ETH/USD|2025-02-04|2883.1|2891.4|2633.9|2731.4|28393.92306441|
|ETH/USD|2025-02-05|2731.5|2827.5|2700.1|2788.7|18123.51419583|
|ETH/USD|2025-02-06|2788.6|2857.0|2655.3|2687.0|18291.88811049|
|ETH/USD|2025-02-07|2687.4|2798.5|2564.2|2623.5|11100.25994478|
|ETH

In [4]:
# =========================================
# NULL CHECK
# =========================================

print("NULL CHECK")

df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show()

# =========================================
# DUPLICATE CHECK
# =========================================

total_rows = df.count()

unique_rows = df.dropDuplicates(
    ["symbol", "timestamp"]
).count()

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicates:", total_rows - unique_rows)

NULL CHECK
+------+---------+----+----+---+-----+------+
|symbol|timestamp|open|high|low|close|volume|
+------+---------+----+----+---+-----+------+
|     0|        0|   0|   0|  0|    0|     0|
+------+---------+----+----+---+-----+------+

Total Rows: 7000
Unique Rows: 7000
Duplicates: 0


In [5]:
# =========================================
# STANDARDIZE TIMESTAMP
# =========================================

df = df.withColumn(
    "timestamp",
    F.to_timestamp("timestamp")
)

df = df.orderBy(
    "symbol",
    "timestamp"
)

df.select(
    F.min("timestamp").alias("min_time"),
    F.max("timestamp").alias("max_time")
).show(truncate=False)

+-------------------+-------------------+
|min_time           |max_time           |
+-------------------+-------------------+
|2025-01-27 00:00:00|2026-06-10 00:00:00|
+-------------------+-------------------+



In [ ]:
# =========================================
# GAP DETECTION
# =========================================

w = Window.partitionBy(
    "symbol"
).orderBy(
    "timestamp"
)

df = df.withColumn(
    "prev_time",
    F.lag("timestamp").over(w)
)

df = df.withColumn(
    "diff_day",
    F.datediff(
        F.col("timestamp"),
        F.col("prev_time")
    )
)

df.select(
    "symbol",
    "timestamp",
    "prev_time",
    "diff_day"
).show(20, False)

gap_count = df.filter(
    F.col("diff_day") > 1
).count()

print("Gap Count:", gap_count)